# Field-line tracing from an NF2 file

This notebook starts from an existing `extrapolation_result.nf2`, detects its geometry, computes Q, twist, integrated current, length, connectivity, and apex diagnostics on one layer, and traces a small set of complete paths.

A CUDA GPU is strongly recommended. Begin with coarse sampling and increase the resolution only after checking convergence and the fraction of valid traces.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from matplotlib.colors import BoundaryNorm, ListedColormap

import nf2

## Configuration

Set `NF2_PATH` to the checkpoint you want to analyze. `step_size_Mm` is physical; NF2 converts it to the checkpoint's normalized model coordinates. `batch_size` counts original seeds, with forward and backward half-lines sharing the live queue.

In [ ]:
NF2_PATH = Path("/path/to/extrapolation_result.nf2")

# Cartesian layer controls.
CARTESIAN_HEIGHT = 0 * u.Mm
CARTESIAN_PIXEL_SIZE = 2.0  # Mm

# Spherical layer controls. Use the inner boundary for topology or, for
# example, 1.4 * u.solRad for an S-web layer.
SPHERICAL_RADIUS = None
SPHERICAL_SAMPLING = (90, 180)  # latitude, longitude samples

TRACE_CONFIG = {
    "method": "rk4",
    "step_size_Mm": 0.25,
    "max_steps": 10_000,
    "batch_size": 2**14,
    "q_method": "tangent",
    "progress": True,
}

assert NF2_PATH.exists(), f"NF2 file not found: {NF2_PATH}"

## Load the checkpoint and sample a layer

The public loader reads the checkpoint once and returns `CartesianOutput` or `SphericalOutput`. All requested line quantities share one central forward/backward trace. The perturbed multi-point stencil is used only when Q is requested with `q_method='perturbed'`.

In [ ]:
out = nf2.load(NF2_PATH)
print(f"Loaded {type(out).__name__} on {out.device}")

requested_metrics = [
    "squashing_factor",
    "twist_number",
    "fieldline_length",
    "integrated_current_density",
    "fieldline_geometry",
]

if isinstance(out, nf2.CartesianOutput):
    geometry = "cartesian"
    layer = out.load_slice(
        z=CARTESIAN_HEIGHT,
        Mm_per_pixel=CARTESIAN_PIXEL_SIZE,
        metrics=requested_metrics,
        trace_config=TRACE_CONFIG,
        compute_jacobian=False,
    )
else:
    geometry = "spherical"
    radius = out.radius_range[0] if SPHERICAL_RADIUS is None else SPHERICAL_RADIUS
    layer = out.load_spherical_layer(
        radius=radius,
        sampling=SPHERICAL_SAMPLING,
        metrics=requested_metrics,
        trace_config=TRACE_CONFIG,
        compute_jacobian=False,
    )

print(f"Geometry: {geometry}")
print(f"Layer shape: {layer['b'].shape[:-1]}")
print(f"Metric keys: {sorted(layer['metrics'])}")

## Prepare maps

Cartesian loader arrays are ordered `(x, y)` and are transposed for `imshow`. Spherical layer arrays are ordered `(colatitude, longitude)` and are flipped into increasing latitude. Invalid Q and incomplete apex traces are masked.

In [ ]:
metrics = layer["metrics"]
q_valid = metrics["q_valid"]
log_q = metrics["log10_q"].copy()
log_q[~q_valid] = np.nan
twist = metrics["twist_number"].value
length = metrics["fieldline_length"].to_value(u.Mm)
integrated_current = metrics["integrated_current_density"]
current_norm = np.linalg.norm(integrated_current, axis=-1)
open_ = metrics["open"]
closed = metrics["closed"]
complete = open_ | closed

if geometry == "cartesian":
    normal_field = layer["b"][..., 2].to_value(u.G)
    apex = metrics["apex_height"].to_value(u.Mm)
    apex_label = r"Apex height [Mm]"
    coords = layer["coords"]
    extent = [coords[..., 0].min(), coords[..., 0].max(),
              coords[..., 1].min(), coords[..., 1].max()]
    xlabel, ylabel = "x [Mm]", "y [Mm]"
    orient = lambda array: np.asarray(array).T
else:
    normal_field = layer["b_rtp"][..., 0].to_value(u.G)
    apex = metrics["apex_radius"].to_value(u.solRad)
    apex_label = r"Apex radius [$R_\odot$]"
    extent = [0, 360, -90, 90]
    xlabel, ylabel = "Carrington longitude [deg]", "Latitude [deg]"
    orient = lambda array: np.flip(np.asarray(array), axis=0)

signed_log_q = np.sign(normal_field) * log_q
apex = apex.astype(float, copy=True)
apex[~complete] = np.nan
connectivity = np.full(open_.shape, np.nan)
connectivity[closed] = 0
connectivity[open_] = 1

signed_log_q = orient(signed_log_q)
twist = orient(twist)
length = orient(length)
current_values = orient(current_norm.value)
connectivity = orient(connectivity)
apex = orient(apex)

In [ ]:
def percentile_limit(values, percentile=99, symmetric=False):
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size == 0:
        return 1.0
    values_for_limit = np.abs(finite) if symmetric else finite
    limit = float(np.nanpercentile(values_for_limit, percentile))
    return limit if limit > 0 else 1.0

q_limit = percentile_limit(signed_log_q, symmetric=True)
twist_limit = percentile_limit(twist, symmetric=True)
current_limit = percentile_limit(current_values)
length_limit = percentile_limit(length)

connectivity_cmap = ListedColormap(["royalblue", "darkorange"])
connectivity_cmap.set_bad("lightgray")
connectivity_norm = BoundaryNorm([-0.5, 0.5, 1.5], connectivity_cmap.N)

fig, axes = plt.subplots(3, 2, figsize=(14, 12), constrained_layout=True)
images = [
    axes[0, 0].imshow(signed_log_q, origin="lower", extent=extent, cmap="bwr",
                      vmin=-q_limit, vmax=q_limit),
    axes[0, 1].imshow(twist, origin="lower", extent=extent, cmap="Spectral_r",
                      vmin=-twist_limit, vmax=twist_limit),
    axes[1, 0].imshow(current_values, origin="lower", extent=extent, cmap="magma",
                      vmin=0, vmax=current_limit),
    axes[1, 1].imshow(length, origin="lower", extent=extent, cmap="viridis",
                      vmin=0, vmax=length_limit),
    axes[2, 0].imshow(connectivity, origin="lower", extent=extent,
                      cmap=connectivity_cmap, norm=connectivity_norm),
    axes[2, 1].imshow(apex, origin="lower", extent=extent, cmap="cividis"),
]
labels = [
    r"$\mathrm{sign}(B_n)\log_{10}Q$",
    r"$T_w$",
    rf"$|\int \mathbf{{J}}\,dl|$ [{current_norm.unit.to_string('latex_inline')}]",
    "Field-line length [Mm]",
    "Connectivity",
    apex_label,
]
for axis, image, label in zip(axes.flat, images, labels):
    axis.set(xlabel=xlabel, ylabel=ylabel)
    if label == "Connectivity":
        colorbar = fig.colorbar(image, ax=axis, ticks=[0, 1])
        colorbar.ax.set_yticklabels(["closed", "open"])
    else:
        fig.colorbar(image, ax=axis, label=label)
plt.show()

## Quality summary

A Q map should always be interpreted together with `q_valid`. Open and closed classifications require both half-lines to terminate at recognized boundaries. Lines stopped by weak field, non-finite coordinates, maximum steps, or maximum length remain incomplete.

In [ ]:
total = q_valid.size
print(f"Valid Q: {q_valid.sum():,}/{total:,} ({q_valid.mean():.1%})")
print(f"Open:    {open_.sum():,}/{total:,} ({open_.mean():.1%})")
print(f"Closed:  {closed.sum():,}/{total:,} ({closed.mean():.1%})")
print(f"Other/incomplete: {(~complete).sum():,}/{total:,} ({(~complete).mean():.1%})")

## Trace selected complete paths

Path recording is intentionally disabled for maps because it is memory intensive. The next cell selects a small set of layer seeds, converts the loader coordinates back to normalized model coordinates, and enables `store_path` only for those lines.

In [ ]:
number_of_lines = 12
flat_coords = layer["coords"].reshape(-1, 3)
complete_flat = complete.reshape(-1)
candidate_indices = np.flatnonzero(complete_flat)
if candidate_indices.size == 0:
    candidate_indices = np.arange(flat_coords.shape[0])
selected = candidate_indices[np.linspace(0, candidate_indices.size - 1,
                                         min(number_of_lines, candidate_indices.size),
                                         dtype=int)]
selected_coords = flat_coords[selected]

if geometry == "cartesian":
    seeds_model = selected_coords / out.Mm_per_ds
    path_scale = out.Mm_per_ds
    path_unit = "Mm"
else:
    solar_radius_per_model_unit = (out.m_per_ds / (1 * u.solRad)).to_value(
        u.dimensionless_unscaled
    )
    seeds_model = selected_coords / solar_radius_per_model_unit
    path_scale = solar_radius_per_model_unit
    path_unit = r"$R_\odot$"

path_config = {**TRACE_CONFIG, "store_path": True}
lines = out.trace(
    seeds_model,
    metrics=["fieldline_length", "fieldline_geometry"],
    trace_config=path_config,
)
print(f"Stored path array: {lines['path'].shape}")

In [ ]:
paths = lines["path"] * path_scale
fig = plt.figure(figsize=(9, 8))
axis = fig.add_subplot(projection="3d")
for index in range(paths.shape[1]):
    path = paths[:, index]
    path = path[np.isfinite(path).all(axis=-1)]
    axis.plot(path[:, 0], path[:, 1], path[:, 2], linewidth=1)
axis.set(
    xlabel=f"x [{path_unit}]",
    ylabel=f"y [{path_unit}]",
    zlabel=f"z [{path_unit}]",
    title="Selected NF2 field lines",
)
plt.show()

## Next steps

Repeat the layer calculation with a smaller `step_size_Mm` and compare Q, twist, and valid masks before producing final maps. For spherical S-web work, set `SPHERICAL_RADIUS = 1.4 * u.solRad`. For perturbed-point Q validation, set `q_method='perturbed'` and choose `q_epsilon_Mm`; expect several times more tracing work. See the dedicated [field-line tracing guide](../../docs/field_line_tracing.md) for definitions, boundaries, status codes, exports, and performance guidance.